# OpsTune — Baseline evaluation

Runs **un-fine-tuned Llama 3.1 8B Instant** (Groq) on the held-out test
split and reports JSON validity, schema compliance, severity/category
accuracy, confidence MAE, and root-cause/action overlap.

This is the *before* number for the hackathon's before/after story.
After the fine-tuning run, swap the model id in the inference cell and
re-run — every metric in this notebook is computed from the cached
predictions file, so re-runs after the LLM calls are instant.

**Required env**: `GROQ_API_KEY` (free tier is fine; `OPENAI_API_KEY`
+ `OPENAI_BASE_URL` also work via the same OpenAI-compatible client).

In [ ]:
from __future__ import annotations

import json
import os
import random
import sys
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "finetuning").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from finetuning.eval.metrics import (
    SEVERITIES, CATEGORIES,
    score_one, aggregate, confusion, label_distribution,
)
from finetuning.generate_reports import SYSTEM_PROMPT

TEST_PATH = ROOT / "finetuning" / "splits" / "test.jsonl"
TEST_LABELED_PATH = ROOT / "finetuning" / "splits" / "test.labeled.jsonl"
PRED_DIR = ROOT / "finetuning" / "eval" / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT: {ROOT}")
print(f"test.jsonl: {TEST_PATH.exists()}, test.labeled.jsonl: {TEST_LABELED_PATH.exists()}")

## Configuration

Change `MODEL_ID` after fine-tuning. Everything else stays the same so
the comparison is apples-to-apples.

In [ ]:
MODEL_ID = os.getenv("BASELINE_MODEL", "llama-3.1-8b-instant")
RUN_NAME = os.getenv("RUN_NAME", "baseline")
PRED_PATH = PRED_DIR / f"{RUN_NAME}.jsonl"

# Generation knobs — keep deterministic for reproducible eval
TEMPERATURE = 0.0
MAX_TOKENS = 700
MAX_WORKERS = 3      # respect Groq free-tier TPM
MAX_RETRIES = 8
BASE_BACKOFF_S = 4.0
MAX_BACKOFF_S = 75.0

print(f"model: {MODEL_ID}\nrun:   {RUN_NAME}\npredictions cache: {PRED_PATH}")

## Load held-out test split

In [ ]:
test_rows = [json.loads(l) for l in TEST_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
test_labeled = [json.loads(l) for l in TEST_LABELED_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
labeled_by_udi = {r["udi"]: r for r in test_labeled}

print(f"{len(test_rows)} test rows")
print("per-mode counts:", dict(Counter(r["metadata"]["primary_mode"] for r in test_rows)))

## Inference (cached on disk)

Each (UDI, MODEL_ID) pair is called once and appended to `predictions/<run>.jsonl`.
Re-running this cell skips UDIs that already have a cached prediction —
safe after rate-limit kills, restarts, or model swaps (use a different `RUN_NAME`).

In [ ]:
from openai import OpenAI

groq_key = os.getenv("GROQ_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")
if groq_key and not openai_key:
    client = OpenAI(api_key=groq_key, base_url="https://api.groq.com/openai/v1")
elif openai_key:
    client = OpenAI(api_key=openai_key, base_url=os.getenv("OPENAI_BASE_URL"))
else:
    raise RuntimeError("Set GROQ_API_KEY or OPENAI_API_KEY")

import re
_RETRY_HINT_RE = re.compile(r"try again in ([\d.]+)s")

def predict(narrative: str) -> str:
    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=MODEL_ID,
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": narrative},
                ],
            )
            return (resp.choices[0].message.content or "").strip()
        except Exception as exc:
            last_err = exc
            wait = min(MAX_BACKOFF_S, BASE_BACKOFF_S * (2 ** attempt)) + random.random()
            m = _RETRY_HINT_RE.search(str(exc))
            if m:
                try: wait = max(wait, float(m.group(1)) + 1.0)
                except ValueError: pass
            time.sleep(wait)
    raise RuntimeError(f"giving up after {MAX_RETRIES} attempts: {last_err}")


def already_cached(path: Path) -> set[int]:
    if not path.exists(): return set()
    seen = set()
    with path.open() as f:
        for line in f:
            try: seen.add(int(json.loads(line)["udi"]))
            except Exception: continue
    return seen

done = already_cached(PRED_PATH)
pending = [r for r in test_rows if r["metadata"]["udi"] not in done]
print(f"{len(done)} cached, {len(pending)} to call")

if pending:
    out_f = PRED_PATH.open("a", encoding="utf-8")
    completed = 0
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(predict, r["messages"][1]["content"]): r for r in pending}
        for fut in as_completed(futures):
            row = futures[fut]
            try: raw = fut.result()
            except Exception as exc:
                raw = f"<<ERROR: {exc}>>"
            out_f.write(json.dumps({
                "udi": row["metadata"]["udi"],
                "primary_mode": row["metadata"]["primary_mode"],
                "raw": raw,
            }, ensure_ascii=False) + "\n")
            out_f.flush()
            completed += 1
            if completed % 10 == 0 or completed == len(pending):
                rate = completed / max(1e-6, time.time() - t0)
                print(f"  {completed}/{len(pending)}  ({rate:.1f} rows/s)")
    out_f.close()

print(f"predictions → {PRED_PATH}")

## Score predictions

In [ ]:
preds_by_udi = {}
with PRED_PATH.open() as f:
    for line in f:
        obj = json.loads(line)
        preds_by_udi[int(obj["udi"])] = obj

scored = []
truths = []
rows_by_udi = {r["metadata"]["udi"]: r for r in test_rows}
for udi, row in rows_by_udi.items():
    truth = json.loads(row["messages"][2]["content"])
    pred_obj = preds_by_udi.get(udi)
    raw = pred_obj["raw"] if pred_obj else ""
    s = score_one(raw, truth)
    s["udi"] = udi
    s["primary_mode"] = row["metadata"]["primary_mode"]
    s["narrative"] = row["messages"][1]["content"]
    s["truth"] = truth
    s["raw"] = raw
    scored.append(s)
    truths.append(truth)

agg = aggregate(scored)
pd.DataFrame({"metric": list(agg.keys()), "value": list(agg.values())})

## Severity & category confusion

In [ ]:
def show_confusion(field, labels):
    mat = confusion(scored, truths, field, labels)
    df = pd.DataFrame(mat, index=[f"true:{l}" for l in labels], columns=[f"pred:{l}" for l in labels])
    print(f"\n{field} confusion (off-vocab predictions excluded):")
    print(df.to_string())
    fig, ax = plt.subplots(figsize=(1.0 + 0.6*len(labels), 1.0 + 0.5*len(labels)))
    im = ax.imshow(mat, cmap="Blues")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(field)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, mat[i][j], ha="center", va="center",
                    color="white" if mat[i][j] > max(max(r) for r in mat)/2 else "black")
    plt.tight_layout(); plt.show()

show_confusion("severity", SEVERITIES)
show_confusion("category", CATEGORIES)

print("\npredicted-severity distribution:", dict(label_distribution(scored, "severity")))
print("predicted-category distribution:", dict(label_distribution(scored, "category")))

## Per-mode breakdown

Where does the base model do best/worst? Big spreads here are the
headline before/after story for the demo.

In [ ]:
df_scored = pd.DataFrame(scored)
by_mode = (df_scored.assign(primary_mode=lambda d: d["primary_mode"].fillna("HEALTHY"))
                     .groupby("primary_mode")
                     .agg(n=("udi","count"),
                          json_valid=("json_valid","mean"),
                          schema_ok=("schema_ok","mean"),
                          severity_acc=("severity_match","mean"),
                          category_acc=("category_match","mean"),
                          conf_mae=("confidence_mae","mean"),
                          causes_jaccard=("causes_jaccard","mean"),
                          actions_jaccard=("actions_jaccard","mean"))
                     .round(3))
by_mode

## Spot-check predictions

In [ ]:
def show_example(s):
    print(f"--- UDI={s['udi']} primary={s['primary_mode']} ---")
    print(f"narrative: {s['narrative']}\n")
    print(f"raw model output:\n{s['raw'][:800]}{'...' if len(s['raw'])>800 else ''}\n")
    print(f"json_valid={s['json_valid']} schema_ok={s['schema_ok']}")
    if s['schema_issues']:
        print(f"  issues: {s['schema_issues']}")
    print(f"severity:  pred={(s['predicted'] or {}).get('severity')!r:>10}  true={s['truth']['severity']!r}")
    print(f"category:  pred={(s['predicted'] or {}).get('category')!r:>10}  true={s['truth']['category']!r}")
    print(f"confidence MAE: {s['confidence_mae']}")
    print(f"causes Jaccard:  {s['causes_jaccard']}")
    print(f"actions Jaccard: {s['actions_jaccard']}\n")

valid = [s for s in scored if s["json_valid"]]
if valid:
    valid_sorted = sorted(valid, key=lambda s: ((s.get("causes_jaccard") or 0) + (s.get("actions_jaccard") or 0)))
    print("### WORST (lowest causes+actions overlap)")
    show_example(valid_sorted[0])
    print("### BEST (highest overlap)")
    show_example(valid_sorted[-1])
broken = [s for s in scored if not s["json_valid"]]
if broken:
    print("### BROKEN JSON example")
    show_example(broken[0])

## Save run summary

Persists the headline numbers so the fine-tuned run can be diffed against
this baseline without re-running anything.

In [ ]:
summary = {
    "run_name": RUN_NAME,
    "model": MODEL_ID,
    "n_test": len(test_rows),
    "aggregate": agg,
    "by_mode": by_mode.reset_index().to_dict(orient="records"),
    "predicted_severity_distribution": dict(label_distribution(scored, "severity")),
    "predicted_category_distribution": dict(label_distribution(scored, "category")),
}
summary_path = PRED_DIR / f"{RUN_NAME}.summary.json"
summary_path.write_text(json.dumps(summary, indent=2, default=str))
print(f"summary → {summary_path}")